# ConnectomeDB human relation context export

This notebook calls the local API relation-context export endpoint, downloads the ZIP bundle, opens the Parquet tables with pandas, and prints the first rows.


In [ ]:
from pathlib import Path
import json
import zipfile

import pandas as pd
import requests


## Run the export request

Make sure the local API is running, e.g. from the repository root:

```bash
make restart DATA_DIR=../omnipath_build/data/combined/latest
```


In [ ]:
api_url = "http://localhost:8081/exports/relation-context/zip"
output_zip = Path("connectomedb_human_interactions.zip")
extract_dir = Path("connectomedb_human_interactions")

payload = {
    "filters": {
        "taxonomy_ids": ["9606"],
        "relation_categories": ["interaction"],
        "sources": ["connectomedb"],
    },
    "require_both_participants_in_entity_scope": True,
    "include_annotations": True,
    "include_evidence": True,
    "filename": "connectomedb_human_interactions",
}

response = requests.post(api_url, json=payload, timeout=120)
response.raise_for_status()
output_zip.write_bytes(response.content)

print(f"Downloaded {output_zip} ({output_zip.stat().st_size:,} bytes)")
print("Response counts:")
for header in ["x-export-relations-count", "x-export-entities-count", "x-export-annotations-count"]:
    print(f"  {header}: {response.headers.get(header)}")


## Extract and inspect bundle contents

In [ ]:
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(output_zip) as zf:
    zf.extractall(extract_dir)
    print("Bundle files:")
    for name in zf.namelist():
        print(f"  {name}")

manifest = json.loads((extract_dir / "manifest.json").read_text())
print("\nManifest:")
print(json.dumps(manifest, indent=2))


## Open tables with pandas

In [ ]:
relations = pd.read_parquet(extract_dir / "relations.parquet")
entities = pd.read_parquet(extract_dir / "entities.parquet")
annotations = pd.read_parquet(extract_dir / "annotations.parquet")

print("relations", relations.shape)
print("entities", entities.shape)
print("annotations", annotations.shape)


## First rows

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Relations:")
display(relations.head())

print("Entities:")
display(entities.head())

print("Annotations:")
display(annotations.head())


## Evidence preview

Evidence is nested inside `relations.parquet` as a list column.

In [ ]:
if "evidence" in relations.columns:
    print("Evidence items in first relations:")
    print(relations["evidence"].head().map(lambda value: len(value) if value is not None else 0))
    print("\nFirst relation evidence:")
    display(pd.DataFrame(relations.loc[0, "evidence"]))
else:
    print("No evidence column found. Set include_evidence=true in the request.")
